# Clasificacion de Diagnostico de Cancer de Mama con Machine Learning

Este analisis utiliza el dataset Wisconsin Breast Cancer para construir modelos de
clasificacion que permitan distinguir entre tumores benignos y malignos a partir de
caracteristicas extraidas de imagenes digitalizadas de biopsias por aspiracion con
aguja fina (FNA). Se exploran tecnicas de preprocesamiento, manejo de desbalance de
clases y multiples algoritmos de clasificacion.


In [ ]:
import seaborn as sns
from sklearn.decomposition import PCA, TruncatedSVD
import matplotlib.patches as mpatches
import time

# Librerias de clasificadores
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
import collections


# Otras Librerias
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from imblearn.pipeline import make_pipeline as imbalanced_make_pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss
from imblearn.metrics import classification_report_imbalanced
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score, classification_report
from collections import Counter
from sklearn.model_selection import KFold, StratifiedKFold
import warnings
warnings.filterwarnings("ignore")

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
%matplotlib inline
import matplotlib.pyplot as plt
import scipy.stats as stats
import xgboost as xgb
from sklearn.model_selection import KFold
from IPython.display import HTML, display
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('data/data.csv')


## 1. Analisis Exploratorio de Datos (EDA)


Comenzamos con un analisis exploratorio para comprender la estructura del dataset,
identificar valores faltantes, detectar registros duplicados y obtener una vision
general de las distribuciones de las variables.


In [ ]:
# Convertimos diagnosis de B/M a 0/1
df['diagnosis'] = df['diagnosis'].map({'B': 0, 'M': 1})
df['diagnosis'].value_counts()

In [ ]:
# definimos "y" como la variable diagnostico 
#definimos a "X" como el resto de variables sin diagnostico, unnamed:32, id

y = df.diagnosis # M or B 
list_drp = ['Unnamed: 32','id','diagnosis']
X = df.drop(list_drp,axis = 1 )


In [ ]:
#cols_with_missing = [col for col in train.columns if train[col].isnull().any()]
cols_with_missing = df.isnull().sum()
cols_with_missing = cols_with_missing[cols_with_missing>0]
cols_with_missing.sort_values(inplace=True)
fig, ax = plt.subplots(figsize=(7,6))  
width = 0.70 # the width of the bars 
ind = np.arange(len(cols_with_missing))  # the x locations for the groups
ax.barh(ind, cols_with_missing, width, color="blue")
ax.set_yticks(ind+width/2)
ax.set_yticklabels(cols_with_missing.index, minor=False)
plt.xlabel('Count')
plt.ylabel('Features') 

In [ ]:
#vemos que unnamed esta vacía, por lo que la sacamos de la base de datos
lista=['Unnamed: 32','id']
df=df.drop(lista,axis = 1 )

In [ ]:
# definimos "y" como la variable diagnostico 
#definimos a "X" como el resto de variables sin diagnostico, unnamed:32, id

y = df.diagnosis # M or B 
list_drp = ['diagnosis']
X = df.drop(list_drp,axis = 1 )





print('Los casos benignos son el', round(df['diagnosis'].value_counts()[0]/len(df) * 100,2), '% del dataset')
print('Los casos malignos son el', round(df['diagnosis'].value_counts()[1]/len(df) * 100,2), '% del dataset')
#graficamos el número de caos benignos y malignos, y de esta forma verificar si la clase es balanceada
ax = sns.countplot(y,label="Count")       # M = 212, B = 357
B, M = y.value_counts()
print('Número de benignos: ',B)
print('Número de malignos : ',M)
ax.set_ylabel('Número de pacientes')
bars = ax.patches
half = int(len(bars)/2)
left_bars = bars[:half]
right_bars = bars[half:]
for left, right in zip(left_bars, right_bars):
    height_l = left.get_height()
    height_r = right.get_height()
    total = height_l + height_r
    ax.text(left.get_x() + left.get_width()/2., height_l + 40, '{0:.0%}'.format(height_l/total), ha="center")
    ax.text(right.get_x() + right.get_width()/2., height_r + 40, '{0:.0%}'.format(height_r/total), ha="center")

In [ ]:
# Vemos si existen registros duplicados
dups = X.duplicated()
# hacemos print de los registros duplicados si es que existen
print(dups.any())
print(X[dups])

In [ ]:
#Hacemos un gráfico de las variables para ver las estadísticas descriptivas de cada una
#también nos permite ver que variables estan 
X.describe().T.style.bar(subset=['mean'], color='#205ff2')\
                            .background_gradient(subset=['std'], cmap='Reds')\
                            .background_gradient(subset=['50%'], cmap='coolwarm')

### Mapa de Calor de Correlaciones

El mapa de calor nos permite visualizar la matriz de correlacion entre todas las
variables continuas. Los colores representan la magnitud y direccion de la
correlacion: valores cercanos a 1 (o -1) indican alta correlacion positiva
(o negativa), mientras que valores cercanos a 0 sugieren poca relacion lineal.


In [ ]:
corrmat = X.corr()
f, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corrmat, annot=True, linewidths=.5, fmt= '.1f',ax=ax);

### Interpretacion de Correlaciones y Seleccion de Variables

El mapa de calor revela grupos de variables altamente correlacionadas entre si.
Por ejemplo, `radius_mean`, `perimeter_mean` y `area_mean` presentan alta
correlacion mutua, lo cual es esperable dado que las tres describen el tamano
del nucleo celular. De manera similar, `compactness_mean`, `concavity_mean` y
`concave points_mean` estan fuertemente correlacionadas.

Para reducir la multicolinealidad y mejorar la estabilidad de los modelos,
seleccionamos un representante de cada grupo de variables correlacionadas,
eliminando las redundantes.


In [ ]:
drop_list1 = ['perimeter_mean','radius_mean','compactness_mean','concave points_mean','radius_se','perimeter_se','radius_worst','perimeter_worst','compactness_worst','concave points_worst','compactness_se','concave points_se','texture_worst','area_worst']
x_1 = X.drop(drop_list1,axis = 1 ) 
#correlation map
f,ax = plt.subplots(figsize=(14, 14))
sns.heatmap(x_1.corr(), annot=True, linewidths=.5, fmt= '.1f',ax=ax)

## 2. Preprocesamiento

Aplicamos un escalado MinMaxScaler para normalizar las variables al rango [0, 1].
Esto es particularmente importante para algoritmos sensibles a la escala de las
variables, como SVM. Ademas, preparamos los conjuntos de entrenamiento y prueba.


In [ ]:
# visualize a minmax scaler transform of the sonar dataset
from pandas import read_csv
from pandas import DataFrame
from pandas.plotting import scatter_matrix
from sklearn.preprocessing import MinMaxScaler
from matplotlib import pyplot

data = df.values[:, :]
# perform a robust scaler transform of the dataset
trans = MinMaxScaler()
data = trans.fit_transform(data)
# convert the array back to a dataframe
dfbalanceado = DataFrame(data)
dfbalanceado.columns = df.columns
# summarize
print(dfbalanceado.describe())
# histograms of the variables
dfbalanceado.hist()
pyplot.show()

In [ ]:
#Este código colocamos la variable diagnosis en la última columna por comodidad
diagnosis = dfbalanceado['diagnosis']

dfbalanceado.drop(['diagnosis'], axis=1, inplace=True)
dfbalanceado.insert(30, 'diagnosis', diagnosis)


dfbalanceado.head()

In [ ]:
print(dfbalanceado.columns
      )

In [ ]:
print(len(dfbalanceado.columns))
print(len(dfbalanceado
          ))

## 3. Manejo de Desbalance de Clases

El dataset presenta un desbalance entre las clases benigno y maligno. Para
abordar este problema, evaluamos dos estrategias de remuestreo: sub-muestreo
aleatorio (under-sampling) y sobre-muestreo aleatorio (over-sampling).


In [ ]:
print('Benigno', round(dfbalanceado['diagnosis'].value_counts()[0]/len(dfbalanceado) * 100,2), '% del dataset')
print('Maligno', round(dfbalanceado['diagnosis'].value_counts()[1]/len(dfbalanceado) * 100,2), '% del dataset')

X = dfbalanceado.drop('diagnosis', axis=1)
y = dfbalanceado['diagnosis']

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)

#convirtiendo los datos en array
Xtrain = Xtrain.values
Xtest = Xtest.values
ytrain = ytrain.values
ytest = ytest.values

# Viendo si la distribución de las etiqueta de train y test se distribuyen de manera similar
train_unique_label, train_counts_label = np.unique(ytrain, return_counts=True)
test_unique_label, test_counts_label = np.unique(ytest, return_counts=True)
print('-' * 100)

print('Distribución de las etiquetas: \n')
print(train_counts_label/ len(ytrain))
print(test_counts_label/ len(ytest))
print(len(Xtrain))
print(len(Xtest))
print(len(ytrain))
print(len(ytest))
print(len(dfbalanceado))
print(len(dfbalanceado.columns))
print(len(X))
print(len(y))

### 3.1 Random Under-Sampling

El sub-muestreo aleatorio reduce la clase mayoritaria al tamano de la clase
minoritaria, eliminando observaciones de forma aleatoria. Esto equilibra las
clases pero puede descartar informacion util.


In [ ]:
from imblearn.under_sampling import RandomUnderSampler

# Vamos a mezclar los datos antes de crear las submuestras.

rus = RandomUnderSampler(random_state=0)
rus.fit(Xtrain, ytrain)
X_train_undersampling, y_train_undersampling = rus.fit_resample(Xtrain, ytrain)

df_under_sampling = pd.DataFrame(X_train_undersampling, columns=dfbalanceado.columns[:-1])
df_under_sampling['diagnosis'] = y_train_undersampling
df_under_sampling

### 3.2 Random Over-Sampling

El sobre-muestreo aleatorio duplica observaciones de la clase minoritaria
hasta igualar la clase mayoritaria. Esto preserva toda la informacion
disponible, aunque puede introducir sobreajuste.


In [ ]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=0)
ros.fit(Xtrain, ytrain)
X_train_oversampling, y_train_oversampling = ros.fit_resample(Xtrain, ytrain)

df_over_sampling = pd.DataFrame(X_train_oversampling, columns=dfbalanceado.columns[:-1])
df_over_sampling['diagnosis'] = y_train_oversampling
df_over_sampling

In [ ]:
colors = ["#0101DF", "#DF0101"]
sns.countplot(x='diagnosis', data=df_under_sampling, palette=colors)
plt.title('Clases igualmente distribuidas', fontsize=14)
plt.show()

### 3.3 Matrices de Correlacion Post-Remuestreo

Comparamos las matrices de correlacion del dataset completo, el sub-muestreado
y el sobre-muestreado para verificar que las relaciones entre variables se
mantienen consistentes tras el remuestreo.


In [ ]:
# Hay que asegurarse de usar la submuestra en nuestra correlación
f, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(24,20))

# DataFrame completo
corr = dfbalanceado.corr()
sns.heatmap(corr, cmap='coolwarm_r', annot_kws={'size':20}, ax=ax1)
ax1.set_title("Matriz de correlación con imbalance \n (no se usa como referencia)", fontsize=14)

sub_sample_corr = df_under_sampling.corr()
sns.heatmap(sub_sample_corr, cmap='coolwarm_r', annot_kws={'size':20}, ax=ax2)
ax2.set_title('Matriz de correlación usando submuestra \n (usar como referencia)', fontsize=14)

over_sample_corr = df_over_sampling.corr()
sns.heatmap(over_sample_corr, cmap='coolwarm_r', annot_kws={'size':20}, ax=ax3)
ax3.set_title('Matriz de correlación usando sobremuestra \n (usar como referencia)', fontsize=14)
plt.show()

## 4. Modelamiento y Evaluacion (Under-Sampling)

Entrenamos tres clasificadores sobre el conjunto sub-muestreado: Support Vector
Classifier (SVC), Decision Tree y Naive Bayes. Para cada uno, realizamos
validacion cruzada y optimizacion de hiperparametros mediante GridSearchCV.


In [ ]:
# Se obtiene X e y a partir del nuevo dataset
X_train_undersampling = df_under_sampling.drop('diagnosis', axis=1)
y_train_undersampling = df_under_sampling['diagnosis']

In [ ]:
classifiers = {
    "Support Vector Classifier": SVC(),
    "DecisionTreeClassifier": DecisionTreeClassifier(),
    "Naive_bayes": GaussianNB()
}

In [ ]:
from sklearn.model_selection import cross_val_score

for key, classifier in classifiers.items():
    classifier.fit(X_train_undersampling, y_train_undersampling)
    training_score = cross_val_score(classifier, X_train_undersampling, y_train_undersampling, cv=5)
    print("Clasificador: ", classifier.__class__.__name__, "tiene una precisión del", round(training_score.mean(), 2) * 100, "%")

In [ ]:
#GridSearchCV para encontrar los mejores hiperparámetros

from sklearn.model_selection import GridSearchCV

# Support Vector Classifier
svc_params = {'C': [0.5, 0.7, 0.9, 1], 'kernel': ['rbf', 'poly', 'sigmoid', 'linear']}
grid_svc = GridSearchCV(SVC(), svc_params)
grid_svc.fit(X_train_undersampling, y_train_undersampling)

# mejor modelo SVC  
svc = grid_svc.best_estimator_

# DecisionTree 
tree_params = {"criterion": ["gini", "entropy"], "max_depth": list(range(2,4,1)), 
              "min_samples_leaf": list(range(5,7,1))}
grid_tree = GridSearchCV(DecisionTreeClassifier(), tree_params)
grid_tree.fit(X_train_undersampling, y_train_undersampling)

# mejor modelo árbol de decisión  
tree_clf = grid_tree.best_estimator_

# Naive 
naive_clf = GaussianNB()
naive_clf.fit(X_train_undersampling, y_train_undersampling)

In [ ]:
svc


In [ ]:
tree_clf

In [ ]:
svc_score = cross_val_score(svc, X_train_undersampling, y_train_undersampling, cv=5)
print('Support Vector Classifier Cross Validation Score', round(svc_score.mean() * 100, 2).astype(str) + '%')

tree_score = cross_val_score(tree_clf, X_train_undersampling, y_train_undersampling, cv=5)
print('DecisionTree Classifier Cross Validation Score', round(tree_score.mean() * 100, 2).astype(str) + '%')

naive_score = cross_val_score(naive_clf, X_train_undersampling, y_train_undersampling, cv=5)
print('Naive Bayes Classifier Cross Validation Score', round(naive_score.mean() * 100, 2).astype(str) + '%')

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_predict
# Crea un DataFrame con todos los puntajes y los nombres de los clasificadores.

svc_pred = cross_val_predict(svc, X_train_undersampling, y_train_undersampling, cv=5,
                             method="decision_function")

tree_pred = cross_val_predict(tree_clf, X_train_undersampling, y_train_undersampling, cv=5)

naive_pred = cross_val_predict(naive_clf, X_train_undersampling, y_train_undersampling, cv=5)

In [ ]:
from sklearn.metrics import roc_auc_score

print('Support Vector Classifier: ', roc_auc_score(y_train_undersampling, svc_pred))
print('Decision Tree Classifier: ', roc_auc_score(y_train_undersampling, tree_pred))
print('Naive Bayes Tree Classifier: ', roc_auc_score(y_train_undersampling, naive_pred))

### 4.1 Matriz de Confusion y Metricas (Under-Sampling)

Evaluamos el desempeno de los modelos entrenados con under-sampling sobre el
conjunto de prueba. La matriz de confusion nos permite visualizar los verdaderos
positivos, falsos positivos, verdaderos negativos y falsos negativos de cada
clasificador.


In [ ]:
df_test = pd.DataFrame(Xtest, columns=dfbalanceado.columns[:-1])
df_test['diagnosis'] = ytest


print('Distribución de las clases en el conjunto de datos de submuestra - OVERSAMPLING')
print(df_test['diagnosis'].value_counts())



sns.countplot(x='diagnosis', data=df_test, palette=colors)
plt.title('Clases igualmente distribuidas', fontsize=14)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred_svc = svc.predict(Xtest)
y_pred_tree = tree_clf.predict(Xtest)
y_pred_naive = naive_clf.predict(Xtest)


svc_cf = confusion_matrix(ytest, y_pred_svc)
tree_cf = confusion_matrix(ytest, y_pred_tree)
naive_cf = confusion_matrix(ytest, y_pred_naive)

fig, ax = plt.subplots(2, 2,figsize=(22,12))


sns.heatmap(naive_cf, ax=ax[0][1], annot=True, cmap=plt.cm.copper)
ax[0][1].set_title("Naive Bayes \n Matriz de Confusión", fontsize=14)
ax[0][1].set_xticklabels(['', ''], fontsize=14, rotation=90)
ax[0][1].set_yticklabels(['', ''], fontsize=14, rotation=360)

sns.heatmap(svc_cf, ax=ax[1][0], annot=True, cmap=plt.cm.copper)
ax[1][0].set_title("Suppor Vector Machine \n Matriz de Confusión", fontsize=14)
ax[1][0].set_xticklabels(['', ''], fontsize=14, rotation=90)
ax[1][0].set_yticklabels(['', ''], fontsize=14, rotation=360)

sns.heatmap(tree_cf, ax=ax[1][1], annot=True, cmap=plt.cm.copper)
ax[1][1].set_title("DecisionTree \n Matriz de Confusión", fontsize=14)
ax[1][1].set_xticklabels(['', ''], fontsize=14, rotation=90)
ax[1][1].set_yticklabels(['', ''], fontsize=14, rotation=360)


plt.show()

In [ ]:
y_true = [0, 1, 2, 2, 2]
y_pred = [0, 0, 2, 2, 1]
target_names = ['Benigno', 'Maligno']
print("Reporte clasificación SVC")
print(classification_report(ytest, y_pred_svc, target_names=target_names))
print("")
print("Reporte clasificación Árbol")
print(classification_report(ytest, y_pred_tree, target_names=target_names))
print("")
print("Reporte clasificación Naive Bayes")
print(classification_report(ytest, y_pred_naive, target_names=target_names))

## 5. Modelamiento y Evaluacion (Over-Sampling)

Repetimos el proceso de entrenamiento y evaluacion utilizando el conjunto
sobre-muestreado. Esto nos permite comparar directamente el efecto de cada
estrategia de balanceo sobre el rendimiento de los modelos.


In [ ]:
# Se obtiene X e y a partir del nuevo dataset
df_over_sampling = df_over_sampling.sample(frac = 1)
df_over_sampling_2 = df_over_sampling.iloc[:50000]

X_train_oversampling = df_over_sampling_2.drop('diagnosis', axis=1)
y_train_oversampling = df_over_sampling_2['diagnosis']

In [ ]:
classifiers = {
    "Support Vector Classifier": SVC(),
    "DecisionTreeClassifier": DecisionTreeClassifier(),
    "Naive_bayes": GaussianNB()
}

In [ ]:
for key, classifier in classifiers.items():
    classifier.fit(X_train_oversampling, y_train_oversampling)
    training_score = cross_val_score(classifier, X_train_oversampling, y_train_oversampling, cv=5)
    print("Clasificador: ", classifier.__class__.__name__, "tiene una precisión del", round(training_score.mean(), 2) * 100, "%")

In [ ]:
#GridSearchCV para encontrar los mejores hiperparámetros

# Support Vector Classifier
svc_params = {'C': [0.5, 0.7, 0.9, 1], 'kernel': ['rbf', 'poly', 'sigmoid', 'linear']}
grid_svc = GridSearchCV(SVC(), svc_params)
grid_svc.fit(X_train_oversampling, y_train_oversampling)

# mejor modelo SVC  
svc = grid_svc.best_estimator_

# DecisionTree 
tree_params = {"criterion": ["gini", "entropy"], "max_depth": list(range(2,4,1)), 
              "min_samples_leaf": list(range(5,7,1))}
grid_tree = GridSearchCV(DecisionTreeClassifier(), tree_params)
grid_tree.fit(X_train_oversampling, y_train_oversampling)

# mejor modelo árbol de decisión  
tree_clf = grid_tree.best_estimator_

# Naive 
naive_clf = GaussianNB()
naive_clf.fit(X_train_oversampling, y_train_oversampling)

In [ ]:
svc

In [ ]:
tree_clf


In [ ]:
svc_score = cross_val_score(svc, X_train_oversampling, y_train_oversampling, cv=5)
print('Support Vector Classifier Cross Validation Score', round(svc_score.mean() * 100, 2).astype(str) + '%')

tree_score = cross_val_score(tree_clf, X_train_oversampling, y_train_oversampling, cv=5)
print('DecisionTree Classifier Cross Validation Score', round(tree_score.mean() * 100, 2).astype(str) + '%')

naive_score = cross_val_score(naive_clf, X_train_oversampling, y_train_oversampling, cv=5)
print('Naive Bayes Classifier Cross Validation Score', round(naive_score.mean() * 100, 2).astype(str) + '%')

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_predict
# Crea un DataFrame con todos los puntajes y los nombres de los clasificadores.

svc_pred = cross_val_predict(svc, X_train_oversampling, y_train_oversampling, cv=5,
                             method="decision_function")

tree_pred = cross_val_predict(tree_clf, X_train_oversampling, y_train_oversampling, cv=5)

naive_pred = cross_val_predict(naive_clf, X_train_oversampling, y_train_oversampling, cv=5)

In [ ]:
from sklearn.metrics import roc_auc_score

print('Support Vector Classifier: ', roc_auc_score(y_train_oversampling, svc_pred))
print('Decision Tree Classifier: ', roc_auc_score(y_train_oversampling, tree_pred))
print('Naive Bayes Tree Classifier: ', roc_auc_score(y_train_oversampling, naive_pred))

## 6. Comparacion Final

Evaluamos los modelos entrenados con over-sampling sobre el conjunto de prueba
y comparamos las matrices de confusion y metricas de clasificacion con los
resultados obtenidos mediante under-sampling. Las curvas ROC nos permiten
visualizar el trade-off entre sensibilidad y especificidad para cada modelo.


In [ ]:
df_test = pd.DataFrame(Xtest, columns=dfbalanceado.columns[:-1])
df_test['diagnosis'] = ytest


print('Distribución de las clases en el conjunto de datos de submuestra - OVERSAMPLING')
print(df_test['diagnosis'].value_counts())



sns.countplot(x='diagnosis', data=df_test, palette=colors)
plt.title('Clases igualmente distribuidas', fontsize=14)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred_svc = svc.predict(Xtest)
y_pred_tree = tree_clf.predict(Xtest)
y_pred_naive = naive_clf.predict(Xtest)


svc_cf = confusion_matrix(ytest, y_pred_svc)
tree_cf = confusion_matrix(ytest, y_pred_tree)
naive_cf = confusion_matrix(ytest, y_pred_naive)

fig, ax = plt.subplots(2, 2,figsize=(22,12))


sns.heatmap(naive_cf, ax=ax[0][1], annot=True, cmap=plt.cm.copper)
ax[0][1].set_title("Naive Bayes \n Matriz de Confusión", fontsize=14)
ax[0][1].set_xticklabels(['', ''], fontsize=14, rotation=90)
ax[0][1].set_yticklabels(['', ''], fontsize=14, rotation=360)

sns.heatmap(svc_cf, ax=ax[1][0], annot=True, cmap=plt.cm.copper)
ax[1][0].set_title("Suppor Vector Machine \n Matriz de Confusión", fontsize=14)
ax[1][0].set_xticklabels(['', ''], fontsize=14, rotation=90)
ax[1][0].set_yticklabels(['', ''], fontsize=14, rotation=360)

sns.heatmap(tree_cf, ax=ax[1][1], annot=True, cmap=plt.cm.copper)
ax[1][1].set_title("DecisionTree \n Matriz de Confusión", fontsize=14)
ax[1][1].set_xticklabels(['', ''], fontsize=14, rotation=90)
ax[1][1].set_yticklabels(['', ''], fontsize=14, rotation=360)


plt.show()

In [ ]:
y_true = [0, 1, 2, 2, 2]
y_pred = [0, 0, 2, 2, 1]
target_names = ['Benigno', 'Maligno']
print("Reporte clasificación SVC")
print(classification_report(ytest, y_pred_svc, target_names=target_names))
print("")
print("Reporte clasificación Árbol")
print(classification_report(ytest, y_pred_tree, target_names=target_names))
print("")
print("Reporte clasificación Naive Bayes")
print(classification_report(ytest, y_pred_naive, target_names=target_names))

In [ ]:
svm_fpr,svm_tpr, trhreshold = roc_curve(ytest, y_pred_svc)
plt.plot(svm_fpr,svm_tpr, linestyle='-')

In [ ]:
svm_fpr,svm_tpr, trhreshold = roc_curve(ytest,y_pred_tree )
plt.plot(svm_fpr,svm_tpr, linestyle='-')

In [ ]:
svm_fpr,svm_tpr, trhreshold = roc_curve(ytest,y_pred_naive )
plt.plot(svm_fpr,svm_tpr, linestyle='-')

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(8, 6))
for name, clf in [("SVC", svc), ("Decision Tree", tree_clf), ("Naive Bayes", naive_clf)]:
    RocCurveDisplay.from_estimator(clf, Xtest, ytest, ax=ax, name=name)
ax.set_title("ROC Curves - Comparación de Clasificadores")
plt.show()